# Data Analyst – Customer Churn Exploration

## Project
PerfectPetClub Customer Churn Prediction

## My Responsibilities
- Explore the available datasets.
- Check data quality and missing values.
- Investigate potential features associated with customer churn.
- Collaborate with the Data Engineer and Data Scientist.
- Support model evaluation and visualisation.

In [ ]:
# Step 1: Import the libraries needed for data exploration

import os
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

In [ ]:
# Step 2: Check the project folder and available data files

project_path = os.path.abspath("..")

print("Project folder:", project_path)

raw_data_path = os.path.join(project_path, "data", "raw")

print("\nFiles available in the raw data folder:")

for filename in os.listdir(raw_data_path):
    print("-", filename)

In [ ]:
# Step 3: Load the three CSV files

crm_path = os.path.join(raw_data_path, "PerfectPetClub_CRM_report.csv")
server_path = os.path.join(raw_data_path, "PerfectPetClub_server_log.csv")
survey_path = os.path.join(raw_data_path, "PerfectPetClub_survey.csv")

crm_df = pd.read_csv(crm_path)
server_df = pd.read_csv(server_path)
survey_df = pd.read_csv(survey_path)

print("All three CSV files loaded successfully!")

In [ ]:
# Step 4: Examine dataset sizes and column names

datasets = {
    "CRM Report": crm_df,
    "Server Log": server_df,
    "Survey": survey_df
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])
    print("Column names:", list(df.columns))

In [ ]:
# Step 5: Data quality checks

for name, df in datasets.items():

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    print("\nDataset size:")
    print(df.shape)

In [ ]:
# Step 6: Inspect the SQLite database structure

db_path = os.path.join(raw_data_path, "PerfectPetClub.db")

# Connect to the database
connection = sqlite3.connect(db_path)

# Find all tables
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection
)

print("Tables found in the database:")
print(tables)

# Inspect the columns in each table
for table_name in tables["name"]:

    print("\n" + "=" * 50)
    print("Table:", table_name)
    print("=" * 50)

    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table_name}")',
        connection
    )

    print(schema[["name", "type"]])

connection.close()

In [ ]:
# Step 7: Investigate subscription information

connection = sqlite3.connect(db_path)

# Count records in the main database tables
table_names = [
    "customer",
    "subscription",
    "subscription_type",
    "sale",
    "payment",
    "product_review"
]

print("RECORD COUNTS")
print("=" * 40)

for table_name in table_names:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total FROM "{table_name}"',
        connection
    )

    print(table_name, ":", count.loc[0, "total"])


# Investigate auto-renew values
print("\nAUTO-RENEW DISTRIBUTION")
print("=" * 40)

auto_renew_summary = pd.read_sql_query(
    """
    SELECT
        auto_renew,
        COUNT(*) AS total_subscriptions
    FROM subscription
    GROUP BY auto_renew
    ORDER BY auto_renew
    """,
    connection
)

print(auto_renew_summary)


# Investigate subscription dates
print("\nSUBSCRIPTION DATE SUMMARY")
print("=" * 40)

date_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN start_date IS NULL OR TRIM(start_date) = '' THEN 1 ELSE 0 END) AS missing_start_dates,
        SUM(CASE WHEN end_date IS NULL OR TRIM(end_date) = '' THEN 1 ELSE 0 END) AS missing_end_dates
    FROM subscription
    """,
    connection
)

print(date_summary)

connection.close()

In [ ]:
# Step 8: Investigate subscription dates

connection = sqlite3.connect(db_path)

subscription_dates = pd.read_sql_query(
    """
    SELECT
        auto_renew,
        MIN(start_date) AS earliest_start,
        MAX(start_date) AS latest_start,
        MIN(end_date) AS earliest_end,
        MAX(end_date) AS latest_end,
        COUNT(*) AS total
    FROM subscription
    GROUP BY auto_renew
    ORDER BY auto_renew
    """,
    connection
)

print("SUBSCRIPTION DATE RANGES")
print(subscription_dates)


# Step 9: Investigate delivery performance

delivery_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_sales,
        SUM(CASE
            WHEN delivered_date IS NULL OR TRIM(delivered_date) = ''
            THEN 1 ELSE 0
        END) AS missing_delivery_dates,
        SUM(CASE
            WHEN delivered_date > expected_date
            THEN 1 ELSE 0
        END) AS potentially_late_deliveries
    FROM sale
    """,
    connection
)

print("\nDELIVERY PERFORMANCE")
print(delivery_summary)

connection.close()

In [ ]:
# Step 8: Investigate subscription dates

connection = sqlite3.connect(db_path)

subscription_dates = pd.read_sql_query(
    """
    SELECT
        auto_renew,
        MIN(start_date) AS earliest_start,
        MAX(start_date) AS latest_start,
        MIN(end_date) AS earliest_end,
        MAX(end_date) AS latest_end,
        COUNT(*) AS total
    FROM subscription
    GROUP BY auto_renew
    ORDER BY auto_renew
    """,
    connection
)

print("SUBSCRIPTION DATE RANGES")
print(subscription_dates)


# Step 9: Investigate delivery performance

delivery_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_sales,
        SUM(CASE
            WHEN delivered_date IS NULL OR TRIM(delivered_date) = ''
            THEN 1 ELSE 0
        END) AS missing_delivery_dates,
        SUM(CASE
            WHEN delivered_date > expected_date
            THEN 1 ELSE 0
        END) AS potentially_late_deliveries
    FROM sale
    """,
    connection
)

print("\nDELIVERY PERFORMANCE")
print(delivery_summary)

connection.close()

## Initial Data Exploration Findings

1. Three CSV datasets were identified: CRM, server logs and customer surveys.
2. The SQLite database contains 10 tables, including customer, subscription, sale and product review.
3. The customer and subscription tables each contain 221 records.
4. The subscription table contains 86 records with auto_renew = 0
   and 135 records with auto_renew = 1.
5. No missing values or exact duplicate rows were detected in the
   three CSV datasets.
6. All 221 subscription records have populated start and end dates.

Further investigation is required to confirm the churn definition,
validate date fields and identify useful predictive features.

In [ ]:
# Step 10: Compare auto-renew status with subscription end dates

connection = sqlite3.connect(db_path)

churn_investigation = pd.read_sql_query(
    """
    SELECT
        auto_renew,
        COUNT(*) AS total_subscriptions,

        SUM(
            CASE
                WHEN DATE(end_date) < DATE('now')
                THEN 1 ELSE 0
            END
        ) AS ended_before_today,

        SUM(
            CASE
                WHEN DATE(end_date) >= DATE('now')
                THEN 1 ELSE 0
            END
        ) AS ending_today_or_later

    FROM subscription
    GROUP BY auto_renew
    ORDER BY auto_renew
    """,
    connection
)

print(churn_investigation)

connection.close()

7. Subscription end dates range from January to December 2026.
8. Some subscriptions have future end dates, so auto-renew status
   alone should not automatically be treated as confirmed churn.
9. The sales table contains 23,188 records.
10. There are 1,090 missing delivery dates and 856 potentially
    late deliveries.
11. Further investigation is required to distinguish missing
    delivery information from genuinely late deliveries.

In [ ]:
# Step 11: Check customer-subscription relationships

connection = sqlite3.connect(db_path)

relationship_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_customers,

        COUNT(s.id) AS matched_subscriptions,

        COUNT(DISTINCT c.id) AS unique_customers,

        COUNT(DISTINCT s.id) AS unique_subscriptions,

        SUM(
            CASE WHEN s.auto_renew = 0
            THEN 1 ELSE 0 END
        ) AS customers_with_auto_renew_disabled,

        SUM(
            CASE WHEN s.auto_renew = 1
            THEN 1 ELSE 0 END
        ) AS customers_with_auto_renew_enabled

    FROM customer c

    LEFT JOIN subscription s
        ON c.subscription_id = s.id
    """,
    connection
)

print(relationship_check)

connection.close()

12. There are 86 subscriptions with auto_renew disabled
    and 135 with auto_renew enabled.
13. Of the 86 subscriptions with auto-renew disabled,
    80 have end dates before the current date and 6 have
    future end dates.
14. Ten subscriptions with auto-renew enabled also have
    end dates before the current date.
15. The auto_renew field is a potential source of the
    churn target, subject to confirmation of its meaning.
16. Subscription end dates and auto-renew status should
    be investigated separately.

In [ ]:
# Step 12: Check survey-to-customer matching

connection = sqlite3.connect(db_path)

survey_match = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_survey_records,

        COUNT(c.id) AS matched_customers,

        COUNT(DISTINCT s.email) AS unique_survey_emails,

        COUNT(DISTINCT c.id) AS unique_matched_customers

    FROM (
        SELECT LOWER(TRIM(email)) AS email
        FROM customer_survey
    ) s

    LEFT JOIN (
        SELECT id, LOWER(TRIM(email)) AS email
        FROM customer
    ) c
    ON s.email = c.email
    """,
    connection
)

print(survey_match)

connection.close()

In [ ]:
# Step 12: Match survey emails to customer emails

connection = sqlite3.connect(db_path)

survey_df.to_sql(
    "temporary_survey",
    connection,
    if_exists="replace",
    index=False
)

survey_match = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_survey_records,
        COUNT(c.id) AS matched_customers,
        COUNT(DISTINCT s.email) AS unique_survey_emails,
        COUNT(DISTINCT c.id) AS unique_matched_customers

    FROM temporary_survey s

    LEFT JOIN customer c
      ON LOWER(TRIM(s.email)) = LOWER(TRIM(c.email))
    """,
    connection
)

print(survey_match)

connection.close()

17. All 221 customers have a matching subscription.
18. The customer-subscription relationship is one-to-one
    in the current database.
19. Survey-to-customer matching is being investigated
    using email addresses without displaying personal information.

In [ ]:
# Step 13: Check CRM-to-customer matching

connection = sqlite3.connect(db_path)

crm_df.to_sql(
    "temporary_crm",
    connection,
    if_exists="replace",
    index=False
)

crm_match = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_crm_records,
        COUNT(c.id) AS matched_crm_records,
        COUNT(DISTINCT r.customer_id) AS unique_crm_customers,
        COUNT(DISTINCT c.id) AS unique_matched_customers

    FROM temporary_crm r

    LEFT JOIN customer c
        ON r.customer_id = c.id
    """,
    connection
)

print("CRM CUSTOMER MATCHING")
print(crm_match)

# Investigate CRM event types
print("\nCRM EVENT TYPES")

crm_events = pd.read_sql_query(
    """
    SELECT
        type,
        COUNT(*) AS total_events
    FROM temporary_crm
    GROUP BY type
    ORDER BY total_events DESC
    """,
    connection
)

print(crm_events)

connection.close()

## Customer and Survey Data Integration

20. All 221 survey records matched customer records using
    normalised email addresses.
21. The survey contains 221 unique email addresses.
22. The customer and subscription tables have a complete
    one-to-one relationship.
23. CRM-to-customer matching is being investigated using
    customer IDs.

In [ ]:
# Clean up the two helper tables created during analysis

connection = sqlite3.connect(db_path)

connection.execute("DROP TABLE IF EXISTS temporary_crm")
connection.execute("DROP TABLE IF EXISTS temporary_survey")

connection.commit()
connection.close()

print("Helper tables removed. Original project tables preserved.")

In [ ]:
# Step 14: Analyse CRM events by customer

crm_customer_summary = (
    crm_df.groupby(["customer_id", "type"])
    .size()
    .unstack(fill_value=0)
)

# Make sure both event categories are represented
for event in ["complaint", "query"]:
    if event not in crm_customer_summary.columns:
        crm_customer_summary[event] = 0

print("CRM CUSTOMER EVENT SUMMARY")
print("Number of customers:", len(crm_customer_summary))

print("\nComplaint statistics:")
print(crm_customer_summary["complaint"].describe())

print("\nQuery statistics:")
print(crm_customer_summary["query"].describe())

print("\nCustomers with no CRM events:", 221 - len(crm_customer_summary))

## CRM Customer Behaviour Analysis

24. All 7,585 CRM records matched customer IDs.
25. CRM activity was recorded for 200 of the 221 customers.
26. The CRM report contains 3,837 complaints and 3,748 queries.
27. Average complaints per customer: 19.19.
28. Average queries per customer: 18.74.
29. The median complaint count was 9.5 and the median query
    count was 9.
30. Complaint count, query count and total CRM events are
    potential features for churn prediction.
31. Twenty-one customers had no CRM events.
32. Further analysis is required to determine whether CRM
    activity is associated with customer churn.

In [ ]:
# Step 15: Investigate server login activity

print("SERVER LOG OVERVIEW")
print("Total records:", len(server_df))
print("Unique sessions:", server_df["session_id"].nunique())
print("Unique login values:", server_df["login"].nunique())

print("\nLOGIN VALUE FREQUENCIES")
print(server_df["login"].value_counts(dropna=False))

print("\nDATE RANGE")

server_dates = pd.to_datetime(
    server_df["datetime"],
    errors="coerce"
)

print("Earliest date:", server_dates.min())
print("Latest date:", server_dates.max())
print("Invalid dates:", server_dates.isna().sum())

In [ ]:
# Step 16: Analyse login activity by customer

# Count login records for each customer internally
login_counts = (
    server_df.groupby("login")
    .size()
)

print("CUSTOMER LOGIN ACTIVITY")
print("=" * 40)

print("Number of customers:", login_counts.count())

print("\nLogin count statistics:")
print(login_counts.describe())

print("\nCustomers with fewer than 10 logins:",
      (login_counts < 10).sum())

print("Customers with more than 100 logins:",
      (login_counts > 100).sum())

print("\nTotal login records:", login_counts.sum())

## Server Login Activity Analysis

33. The server log contains 67,249 records.
34. All 67,249 session IDs are unique.
35. Login activity was recorded for 221 unique customers.
36. All server timestamps were successfully parsed.
37. Server activity ranges from 1 January 2026 to
    10 November 2026.
38. Login count per customer is a potential feature
    for customer churn prediction.
39. Further investigation is required to establish
    whether login activity is associated with churn.

## Customer Login Activity Findings

40. Login activity was recorded for all 221 customers.

41. The server log contains 67,249 total login records.

42. The average number of logins per customer is approximately 304.

43. The median number of logins per customer is 198.

44. Login activity varies considerably, ranging from 1 to 1,540
    records per customer.

45. Five customers had fewer than 10 login records.

46. A total of 159 customers had more than 100 login records.

47. Login count may be a useful feature for customer churn prediction.

48. Further analysis is required to determine whether lower login
    activity is associated with churn.

In [ ]:
# Step 17: Analyse how recently customers logged in

server_analysis = server_df.copy()

# Convert the server timestamps into datetime format
server_analysis["datetime_parsed"] = pd.to_datetime(
    server_analysis["datetime"],
    errors="coerce"
)

# Calculate login activity for each customer
login_features = (
    server_analysis.groupby("login")
    .agg(
        login_count=("session_id", "count"),
        first_login=("datetime_parsed", "min"),
        last_login=("datetime_parsed", "max")
    )
)

# Use the latest date in the dataset as the reference date
reference_date = server_analysis["datetime_parsed"].max()

# Calculate days since each customer's last recorded login
login_features["days_since_last_login"] = (
    reference_date - login_features["last_login"]
).dt.days

print("LOGIN RECENCY ANALYSIS")
print("=" * 40)

print("\nDays since last login:")
print(login_features["days_since_last_login"].describe())

print(
    "\nCustomers with no login in the last 7 days:",
    (login_features["days_since_last_login"] > 7).sum()
)

print(
    "Customers with no login in the last 30 days:",
    (login_features["days_since_last_login"] > 30).sum()
)

print("\nCustomers analysed:", len(login_features))

## Login Frequency and Recency Analysis

40. Login activity was recorded for 221 customers.

41. Customers generated 67,249 total login records.

42. The average number of logins per customer was approximately 304.29.

43. The median number of logins per customer was 198.

44. Five customers had fewer than 10 logins, while 159
    customers had more than 100 logins.

45. The average time since the last login was approximately
    58 days, while the median was only 1 day.

46. A total of 81 customers had not logged in during the
    previous 7 days.

47. A total of 72 customers had not logged in during the
    previous 30 days.

48. Login frequency and days since last login are potential
    features for customer churn prediction.

49. Further analysis is required to determine whether
    reduced login activity is associated with churn.